# Experiment 10: Transformer-Based LLM Chatbot Using Gradio

**Aim:** Develop an interactive chatbot using a pre-trained Transformer-based language model.

**Run in Google Colab:** Run the cells from top to bottom. The model downloads the first time. A GPU is helpful but not required. This notebook uses Gradio's `ChatInterface`, avoiding the outdated `gr.Chatbot(type="tuples")` argument that caused the error in the earlier version.

**Dinesh Kumar P - 24BAD021**

In [1]:
!pip -q install -U transformers gradio

In [2]:
!pip uninstall -y torchvision

Found existing installation: torchvision 0.26.0+cpu
Uninstalling torchvision-0.26.0+cpu:
  Successfully uninstalled torchvision-0.26.0+cpu


## Part A — Load and configure a pre-trained LLM



In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "distilgpt2"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.eval()
print("Model loaded successfully on:", device)
print("Dinesh Kumar P - 24BAD021")

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded successfully on: cpu
Dinesh Kumar P - 24BAD021


In [4]:
sample_prompt = "Artificial intelligence is useful because"
inputs = tokenizer(sample_prompt, return_tensors="pt").to(device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))
print("Dinesh Kumar P - 24BAD021")

Artificial intelligence is useful because it is not only able to tell what people are thinking about and what they think about, but also to predict what they think about and what they think about.


















Dinesh Kumar P - 24BAD021


## Part B — Chatbot function and conversation history



In [5]:
def _history_to_text(history):
    parts = []
    for item in (history or [])[-6:]:
        if isinstance(item, dict):
            role = item.get("role", "user")
            content = item.get("content", "")
            if isinstance(content, str) and content.strip():
                label = "Assistant" if role == "assistant" else "User"
                parts.append(f"{label}: {content.strip()}")
        elif isinstance(item, (tuple, list)) and len(item) >= 2:
            user_text, assistant_text = item[0], item[1]
            if user_text:
                parts.append(f"User: {user_text}")
            if assistant_text:
                parts.append(f"Assistant: {assistant_text}")
    return parts


def generate_response(user_message, history=None):
    user_message = str(user_message).strip()
    if not user_message:
        return "Please enter a message."

    prompt_parts = _history_to_text(history)
    prompt_parts.append(f"User: {user_message}\nAssistant:")
    prompt = "\n".join(prompt_parts)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=700
    ).to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    if not answer:
        answer = "I could not generate a useful response. Please try another question."
    return answer

In [6]:
test_questions = ["What is machine learning?", "Why is Python popular?"]
test_history = []
for question in test_questions:
    answer = generate_response(question, test_history)
    print("User:", question)
    print("Bot:", answer, "\n")
    test_history.append({"role": "user", "content": question})
    test_history.append({"role": "assistant", "content": answer})
    print("Dinesh Kumar P - 24BAD021")

User: What is machine learning?
Bot: Machine learning, which means you can be very good at it. The more data comes into your system and the better a model becomes that you have an accurate understanding of how to predict or detect patterns in people's behavior. In other words, when we look at our predictions as real, there are lots of problems with this approach because they're not really going to solve them on their own (or even for any given point). But I think these approaches work pretty well now; if someone goes after one 

Dinesh Kumar P - 24BAD021
User: Why is Python popular?
Bot: Because humans don't like things too much anymore. They just hate new ideas so quickly. That being said, many programmers tend towards systems where machines aren-like - but what does get interesting about artificial intelligence seems to me to be why programming languages generally do worse than those developed by computers today. There isn`t anything wrong with working from day one until tomorrow mor

## Part C — Interactive Gradio chat interface



In [7]:
import gradio as gr

def chat(message, history):
    return generate_response(message, history)

demo = gr.ChatInterface(
    fn=chat,
    title="Transformer-Based LLM Chatbot",
    description="Ask a question below. This small pre-trained model may sometimes produce inaccurate or irrelevant answers.",
    textbox=gr.Textbox(placeholder="Type your question and press Enter", label="Your message")
)

demo.launch(share=True, debug=False)
print("Dinesh Kumar P - 24BAD021")

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a70f5c34729a058c14.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Dinesh Kumar P - 24BAD021


## Part D — Application testing and deployment


In [8]:
requirements_text = "torch\ntransformers\ngradio\n"
app_code = 'import torch\nimport gradio as gr\nfrom transformers import AutoTokenizer, AutoModelForCausalLM\n\nMODEL_NAME = "distilgpt2"\ndevice = "cuda" if torch.cuda.is_available() else "cpu"\ntokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)\nmodel = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)\nif tokenizer.pad_token is None:\n    tokenizer.pad_token = tokenizer.eos_token\nmodel.eval()\n\ndef history_to_text(history):\n    parts = []\n    for item in (history or [])[-6:]:\n        if isinstance(item, dict):\n            role, content = item.get("role", "user"), item.get("content", "")\n            if isinstance(content, str) and content.strip():\n                parts.append(f"{\'Assistant\' if role == \'assistant\' else \'User\'}: {content.strip()}")\n        elif isinstance(item, (tuple, list)) and len(item) >= 2:\n            if item[0]: parts.append(f"User: {item[0]}")\n            if item[1]: parts.append(f"Assistant: {item[1]}")\n    return parts\n\ndef respond(message, history):\n    parts = history_to_text(history)\n    parts.append(f"User: {message.strip()}\\nAssistant:")\n    prompt = "\\n".join(parts)\n    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=700).to(device)\n    with torch.no_grad():\n        output = model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=0.7, top_p=0.9, repetition_penalty=1.1, pad_token_id=tokenizer.eos_token_id, eos_token_id=tokenizer.eos_token_id)\n    answer = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()\n    return answer or "Please try another question."\n\ndemo = gr.ChatInterface(fn=respond, title="Transformer-Based LLM Chatbot", description="A small demo model; responses may be inaccurate.")\ndemo.launch()\n'

with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements_text)
with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("Created app.py and requirements.txt in the current notebook runtime.")
print("requirements.txt:")
print(requirements_text)
print("Dinesh Kumar P - 24BAD021")

Created app.py and requirements.txt in the current notebook runtime.
requirements.txt:
torch
transformers
gradio

Dinesh Kumar P - 24BAD021
